# Lecture 01a: ML Foundations — PyTorch Basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_ML_Engineering/blob/main/lectures/01_a_ml_foundations/exercises/01_pytorch_basics/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_ML_Engineering/main/lectures/01_a_ml_foundations/exercises/01_pytorch_basics/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import (
    test_pairwise_distances, test_einops_flatten,
    test_einops_split_heads, test_einops_mean_pool,
    test_quadratic_grad, test_network_grad,
)

In [ ]:
import torch
from einops import rearrange, reduce, repeat

## Tensor Basics

Tensors are n-dimensional arrays, just like numpy arrays but with GPU support and automatic differentiation. If you know numpy, you already know 90% of this.

In [ ]:
# Creating tensors
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(3, 4)
c = torch.randn(2, 3)       # standard normal
d = torch.arange(0, 10, 2)  # [0, 2, 4, 6, 8]
e = torch.linspace(0, 1, 5) # [0, 0.25, 0.5, 0.75, 1]

print(f"a: {a}")
print(f"shape: {c.shape}, dtype: {c.dtype}, ndim: {c.ndim}")
print(f"arange: {d}")
print(f"linspace: {e}")

In [ ]:
# Element-wise operations (just like numpy)
x = torch.tensor([1.0, 4.0, 9.0])
print(f"sqrt:  {torch.sqrt(x)}")
print(f"exp:   {torch.exp(x)}")
print(f"x + 1: {x + 1}")
print(f"x * 2: {x * 2}")

In [ ]:
# Indexing and boolean masks
M = torch.arange(12).reshape(3, 4).float()
print(f"M:\n{M}")
print(f"Row 0:    {M[0]}")
print(f"Col 2:    {M[:, 2]}")
print(f"M > 5:    {M[M > 5]}")

## Reductions and Broadcasting

**Reductions** collapse a dimension. The `dim` argument says *which* axis to sum/mean over:

In [ ]:
M = torch.arange(12).reshape(3, 4).float()
print(f"M shape: {M.shape}")
print(f"M:\n{M}\n")
print(f"sum over dim=0 (collapse rows):    {M.sum(dim=0)}  shape: {M.sum(dim=0).shape}")
print(f"sum over dim=1 (collapse columns): {M.sum(dim=1)}  shape: {M.sum(dim=1).shape}")
print(f"sum over all:                      {M.sum()}")

**Broadcasting** lets you do operations between tensors of different shapes. PyTorch aligns dimensions from the right and expands size-1 dimensions:

In [ ]:
# (3, 1) * (1, 4) -> (3, 4): outer product
col = torch.tensor([[1.0], [2.0], [3.0]])  # shape (3, 1)
row = torch.tensor([[10.0, 20.0, 30.0, 40.0]])  # shape (1, 4)
print(f"col shape: {col.shape}, row shape: {row.shape}")
print(f"col * row (broadcasted):\n{col * row}")

In [ ]:
# Matrix multiply with @
A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = A @ B  # (3, 5)
print(f"({tuple(A.shape)}) @ ({tuple(B.shape)}) -> {tuple(C.shape)}")

---

## Exercise A: Pairwise Distances Without Loops

Given N points in 2D, compute the N×N matrix of Euclidean distances using broadcasting. No loops.

$$d_{ij} = \|x_i - x_j\|$$

**Hint:** `unsqueeze` to create shapes `(N, 1, 2)` and `(1, N, 2)`, subtract, then take the norm over the last dimension.

In [ ]:
def pairwise_distances(points):
    """
    Args:
        points: (N, 2) tensor of 2D coordinates
    Returns:
        (N, N) tensor of pairwise Euclidean distances
    """
    # TODO (~2 lines): use broadcasting to compute all pairwise distances
    pass

In [ ]:
test_pairwise_distances(pairwise_distances)

<details>
<summary><b>Hint</b></summary>

`points.unsqueeze(0)` has shape `(1, N, 2)` and `points.unsqueeze(1)` has shape `(N, 1, 2)`. Subtracting broadcasts to `(N, N, 2)`. Then use `.norm(dim=-1)`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def pairwise_distances(points):
    diff = points.unsqueeze(0) - points.unsqueeze(1)
    return diff.norm(dim=-1)
```
</details>

In [ ]:
# Visualize your distance matrix
import matplotlib.pyplot as plt
torch.manual_seed(0)
pts = torch.randn(30, 2)
D = pairwise_distances(pts)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(pts[:, 0], pts[:, 1])
ax1.set_title("Points")
ax1.set_aspect("equal")
ax2.imshow(D, cmap="viridis")
ax2.set_title("Pairwise distance matrix")
plt.colorbar(ax2.images[0], ax=ax2)
plt.tight_layout()
plt.show()

---

## Einops: Einstein Notation for Tensor Plumbing

Raw PyTorch reshape/permute chains are hard to read. [Einops](https://einops.rocks/) makes the intention explicit by naming dimensions:

In [ ]:
x = torch.randn(2, 3, 8, 8)  # (batch, channels, height, width)

# Flatten spatial dimensions
flat = rearrange(x, 'b c h w -> b (c h w)')
print(f"Flatten:      {x.shape} -> {flat.shape}")

# Transpose sequence and feature dims
y = torch.randn(2, 10, 64)
transposed = rearrange(y, 'b s d -> b d s')
print(f"Transpose:    {y.shape} -> {transposed.shape}")

# Mean pooling over a sequence
pooled = reduce(y, 'b s d -> b d', 'mean')
print(f"Mean pool:    {y.shape} -> {pooled.shape}")

# Repeat/tile
z = torch.randn(2, 64)
tiled = repeat(z, 'b d -> b n d', n=5)
print(f"Repeat:       {z.shape} -> {tiled.shape}")

The key insight: **einops patterns read like dimension annotations**. Compare:

```python
# PyTorch (what does this do?)
x.reshape(B, S, n_heads, d_head).permute(0, 2, 1, 3)

# Einops (self-documenting)
rearrange(x, 'b s (h d) -> b h s d', h=n_heads)
```

## Exercise B: Einops Translation

Rewrite each operation as a single einops call.

### Part 1: Flatten for a linear layer

Given images of shape `(batch, channels, height, width)`, flatten all non-batch dimensions into a single vector.

In [ ]:
def einops_flatten(x):
    """
    Args:
        x: (batch, channels, height, width) tensor
    Returns:
        (batch, channels*height*width) tensor
    """
    # TODO (1 line): use rearrange
    pass

In [ ]:
test_einops_flatten(einops_flatten)

### Part 2: Split into attention heads

Given `(batch, seq_len, d_model)`, split `d_model` into `n_heads` separate heads and move the head dimension before the sequence dimension. This is the reshape that happens inside every attention layer.

In [ ]:
def einops_split_heads(x, n_heads):
    """
    Args:
        x: (batch, seq_len, d_model) tensor
        n_heads: number of attention heads
    Returns:
        (batch, n_heads, seq_len, d_head) tensor
    """
    # TODO (1 line): use rearrange
    pass

In [ ]:
test_einops_split_heads(einops_split_heads)

### Part 3: Mean pooling

Given token embeddings `(batch, seq_len, d_model)`, compute the mean over the sequence dimension.

In [ ]:
def einops_mean_pool(x):
    """
    Args:
        x: (batch, seq_len, d_model) tensor
    Returns:
        (batch, d_model) tensor
    """
    # TODO (1 line): use reduce
    pass

In [ ]:
test_einops_mean_pool(einops_mean_pool)

<details>
<summary><b>Solutions</b></summary>

```python
# Part 1
def einops_flatten(x):
    return rearrange(x, 'b c h w -> b (c h w)')

# Part 2
def einops_split_heads(x, n_heads):
    return rearrange(x, 'b s (h d) -> b h s d', h=n_heads)

# Part 3
def einops_mean_pool(x):
    return reduce(x, 'b s d -> b d', 'mean')
```
</details>

---

## Autograd: Automatic Differentiation

PyTorch builds a computational graph as you do operations. Calling `.backward()` applies the chain rule through the entire graph and stores gradients on leaf tensors.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 3 * x + 1  # y = x^2 + 3x + 1
y.backward()             # dy/dx = 2x + 3 = 9

print(f"x = {x.item()}")
print(f"y = {y.item()}")
print(f"dy/dx = {x.grad.item()}  (expected: {2 * x.item() + 3})")

In [ ]:
# torch.no_grad() disables tracking (use during inference or manual param updates)
x = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
    y = x * 3  # no graph built
print(f"y.requires_grad: {y.requires_grad}")  # False

# .detach() cuts a tensor out of the graph
z = (x ** 2).detach()
print(f"z.requires_grad: {z.requires_grad}")  # False

## Exercise C: Autograd Detective

Use `requires_grad` and `.backward()` to compute gradients automatically, then check them against what you know analytically.

### Part 1: Quadratic form

For a matrix $A$ and vector $x$, the gradient of $f(x) = x^T A x$ is $\nabla_x f = (A + A^T) x$.

Compute this gradient using autograd.

In [ ]:
def quadratic_grad(A, x):
    """
    Args:
        A: (3, 3) matrix (not necessarily symmetric)
        x: (3,) vector
    Returns:
        (3,) gradient df/dx computed via autograd
    """
    # TODO: clone x with requires_grad, compute f = x^T A x, call backward, return the grad
    pass

In [ ]:
test_quadratic_grad(quadratic_grad)

### Part 2: Gradient through a mini network

Compute the gradient of $f = \sum \text{softmax}(Wx + b)$ with respect to $W$.

The chain rule through softmax is annoying by hand. Autograd just does it.

In [ ]:
def network_grad(W, x, b):
    """
    Args:
        W: (4, 3) weight matrix
        x: (3,) input vector
        b: (4,) bias vector
    Returns:
        (4, 3) gradient df/dW computed via autograd
    """
    # TODO: clone W with requires_grad, compute f = softmax(Wx + b).sum(), backward, return W.grad
    pass

In [ ]:
test_network_grad(network_grad)

<details>
<summary><b>Solutions</b></summary>

```python
# Part 1
def quadratic_grad(A, x):
    x = x.clone().requires_grad_(True)
    f = x @ A @ x
    f.backward()
    return x.grad

# Part 2
def network_grad(W, x, b):
    W = W.clone().requires_grad_(True)
    logits = W @ x + b
    f = torch.softmax(logits, dim=0).sum()
    f.backward()
    return W.grad
```
</details>

---

## Devices

Tensors live on a specific device (CPU or GPU). Moving to GPU enables parallel computation, but GPU memory is the main bottleneck for model/batch size.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

x = torch.randn(3, 3)
x_device = x.to(device)
print(f"x is on: {x_device.device}")

# All tensors in an operation must be on the same device
# y = x.to("cpu") + x.to("cuda")  # would be a RuntimeError!